In [ ]:
!pip -q install sentence-transformers faiss-cpu pypdf gradio datasets ollama
!apt-get -qq update && apt-get -qq install -y pciutils lshw zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time

subprocess.Popen(["ollama", "serve"],
                 stdout=open("/kaggle/working/ollama.log", "w"),
                 stderr=subprocess.STDOUT)
time.sleep(8)

LLM = "qwen2.5:7b"          # nếu thiếu VRAM/chậm: "qwen2.5:3b"
!ollama pull {LLM}

In [ ]:
import ollama
r = ollama.chat(model=LLM, messages=[{"role": "user", "content": "Xin chào, bạn là ai?"}])
print(r["message"]["content"])

In [ ]:
import glob
from datasets import load_dataset
from pypdf import PdfReader

docs = []   # mỗi phần tử: {"head", "body", "src", "lang", "link"}

def pick(cols, cands):
    for c in cands:
        if c in cols:
            return c
    return None

def add_qa(name, config=None, splits=("train",), lang="vi", limit=None,
           q_cands=("question", "Question", "input"),
           a_cands=("answer", "Answer", "output", "long_answer")):
    """Nạp 1 dataset hỏi đáp từ Hugging Face; lỗi thì bỏ qua, không làm hỏng cả pipeline."""
    try:
        dd = load_dataset(name, config) if config else load_dataset(name)
    except Exception as e:
        print(f"❌ Bỏ qua {name}: {str(e)[:150]}")
        return
    n = 0
    for sp in splits:
        if sp not in dd:
            continue
        ds = dd[sp]
        qc, ac = pick(ds.column_names, q_cands), pick(ds.column_names, a_cands)
        if not qc or not ac:
            print(f"❌ {name}: không thấy cột hỏi/đáp. Các cột: {ds.column_names}")
            return
        for row in ds:
            if limit and n >= limit:
                break
            q, a = str(row[qc] or "").strip(), str(row[ac] or "").strip()
            if len(q) < 5 or len(a) < 5:
                continue
            docs.append({"head": f"Q: {q}\nA:", "body": a,
                         "src": name.split("/")[-1], "lang": lang,
                         "link": str(row.get("link", "") or "")})
            n += 1
    print(f"✅ {name}: {n} cặp hỏi đáp")

def add_files(folder="/kaggle/input"):
    """Nạp PDF/TXT/MD do bạn tải lên Kaggle."""
    n = 0
    for f in glob.glob(f"{folder}/**/*.pdf", recursive=True):
        try:
            for i, p in enumerate(PdfReader(f).pages):
                t = (p.extract_text() or "").strip()
                if t:
                    docs.append({"head": "", "body": t, "src": f"{f.split('/')[-1]} - trang {i+1}",
                                 "lang": "vi", "link": ""}); n += 1
        except Exception as e:
            print("Lỗi đọc", f, e)
    for ext in ("txt", "md"):
        for f in glob.glob(f"{folder}/**/*.{ext}", recursive=True):
            t = open(f, encoding="utf-8", errors="ignore").read().strip()
            if t:
                docs.append({"head": "", "body": t, "src": f.split("/")[-1],
                             "lang": "vi", "link": ""}); n += 1
    print(f"✅ File của khoa: {n} đoạn/trang")

# --- Tiếng Việt ---
add_qa("tarudesu/ViHealthQA", splits=("train", "validation", "test"), lang="vi")
add_qa("hungnm/vietnamese-medical-qa", lang="vi")

# --- Tiếng Anh (giữ nguyên bản, LLM sẽ trả lời bằng tiếng Việt) ---
add_qa("keivalya/MedQuad-MedicalQnADataset", lang="en", limit=6000)
add_qa("medalpaca/medical_meadow_wikidoc", lang="en", limit=4000)
add_qa("qiaojin/PubMedQA", config="pqa_labeled", lang="en")

# --- File riêng của khoa ---
add_files()

print("\nTổng:", len(docs), "tài liệu")

In [ ]:
import os, json, faiss, numpy as np
from sentence_transformers import SentenceTransformer

emb = SentenceTransformer("BAAI/bge-m3", device="cuda")
emb.max_seq_length = 512
emb.half()

def split_text(text, size=1000, overlap=150):
    return [text[i:i+size] for i in range(0, max(len(text), 1), size - overlap)]

chunks = []
for d in docs:
    pieces = [d["body"]] if len(d["head"]) + len(d["body"]) <= 1200 else split_text(d["body"])
    for p in pieces:
        chunks.append({"text": (d["head"] + "\n" + p).strip(),
                       "src": d["src"], "lang": d["lang"], "link": d["link"]})
print(len(chunks), "đoạn")

CACHE_X, CACHE_C = "/kaggle/working/X.npy", "/kaggle/working/chunks.json"
if os.path.exists(CACHE_X) and os.path.exists(CACHE_C):
    X = np.load(CACHE_X)
    chunks = json.load(open(CACHE_C, encoding="utf-8"))
    print("Đã nạp cache")
else:
    X = emb.encode([c["text"] for c in chunks], batch_size=64,
                   normalize_embeddings=True, show_progress_bar=True).astype("float32")
    np.save(CACHE_X, X)
    json.dump(chunks, open(CACHE_C, "w", encoding="utf-8"), ensure_ascii=False)

index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
print("Index:", index.ntotal)

In [ ]:
import ollama
from sentence_transformers import CrossEncoder

# Reranker đa ngôn ngữ: chấm lại độ liên quan của các ứng viên lấy từ FAISS
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda")

SYSTEM = """Bạn là trợ lý hỏi đáp y khoa của khoa. Quy tắc:
1. Chỉ trả lời dựa trên NGỮ CẢNH được cung cấp. Nếu ngữ cảnh không đủ thông tin, nói: "Mình chưa tìm thấy thông tin này trong tài liệu."
2. Ngữ cảnh có thể bằng tiếng Anh: hãy đọc hiểu và LUÔN trả lời bằng tiếng Việt tự nhiên. Thuật ngữ y khoa quan trọng ghi kèm tiếng Anh trong ngoặc.
3. NGỮ CẢNH là các hỏi đáp tham khảo của NGƯỜI KHÁC, không phải thông tin về người đang hỏi. Tuyệt đối không gán tuổi, bệnh sử, triệu chứng hay người thân trong ngữ cảnh cho người dùng. Chỉ rút ra kiến thức chung để trả lời đúng câu hỏi.
4. Không tự bịa liều thuốc, chẩn đoán, chỉ số hay số liệu. Chỉ nêu con số nếu nó xuất hiện trong ngữ cảnh.
5. Không mở đầu bằng "Chào bạn" hay "Theo ngữ cảnh". Trả lời thẳng vào câu hỏi, ngắn gọn, ý chính gạch đầu dòng.
6. Kết thúc bằng một câu nhắc: thông tin chỉ mang tính tham khảo, cần hỏi bác sĩ khi có triệu chứng."""

def retrieve(question, k_fetch=20, k_final=5, min_dense=0.3, min_rerank=0.05):
    """B1: FAISS lấy k_fetch ứng viên. B2: reranker chấm lại, giữ k_final đoạn tốt nhất."""
    v = emb.encode([question], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(v, k_fetch)
    cands = [chunks[i] for s, i in zip(scores[0], idx[0]) if i != -1 and s >= min_dense]
    if not cands:
        return []
    rr = reranker.predict([(question, c["text"]) for c in cands], batch_size=16)
    ranked = sorted(zip(cands, rr), key=lambda x: x[1], reverse=True)
    return [(c, float(s)) for c, s in ranked[:k_final] if s >= min_rerank]

def answer_stream(question, debug=False):
    hits = retrieve(question)
    if debug:
        for h, s in hits:
            print(f"[rerank={s:.3f}] {h['src']} | {h['text'][:80]!r}")
    if not hits:
        yield "Mình chưa tìm thấy thông tin này trong tài liệu."
        return

    context = "\n\n".join(f"[{i+1}] (nguồn: {h['src']})\n{h['text']}"
                          for i, (h, _) in enumerate(hits))
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content":
            f"NGỮ CẢNH (hỏi đáp tham khảo của NGƯỜI KHÁC, không phải thông tin về người đang hỏi):\n{context}"
            f"\n\nCÂU HỎI CỦA NGƯỜI DÙNG: {question}\n\nHãy trả lời bằng tiếng Việt."},
    ]

    out = ""
    for part in ollama.chat(model=LLM, messages=messages, stream=True,
                            options={"temperature": 0.2, "num_ctx": 6144}):
        out += part["message"]["content"]
        yield out

    seen, srcs = set(), []
    for h, _ in hits:
        label = h["src"] + (f" ({h['link']})" if h["link"] else "")
        if label not in seen:
            seen.add(label); srcs.append(label)
    yield out + "\n\n**Nguồn:** " + "; ".join(srcs)


In [ ]:
tests = [
    "Các biến chứng thường gặp của đái tháo đường là gì?",      # lỗi cũ: gán "mẹ bạn 55 tuổi" cho người dùng
    "Đang chích ngừa viêm gan B có tiêm vaccine Covid-19 được không?",
    "Cách chế tạo bom?",                                         # ngoài phạm vi: phải nói chưa tìm thấy
]
for q in tests:
    print("=" * 80); print("HỎI:", q)
    for text in answer_stream(q, debug=True):
        pass
    print(text)


In [ ]:
import gradio as gr

def chat_fn(message, history):
    yield from answer_stream(message)

gr.ChatInterface(
    chat_fn,
    title="Chatbot hỏi đáp Y khoa - Khoa",
    description="Thông tin chỉ mang tính tham khảo, không thay thế tư vấn của bác sĩ.",
    examples=["Tiêm vaccine viêm gan B xong có cần kiêng gì không?",
              "Triệu chứng của bệnh đái tháo đường là gì?"],
).launch(share=True, debug=True)